1. Setup and build

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_loader import load_all
from src.data_prep import build_order_table, LEAKAGE_COLS

pd.set_option("display.max_columns", 50)
dfs = load_all()
master = build_order_table(dfs)
print("Raw orders:", len(dfs["orders"]), "| Analytical table:", len(master))
master.head()

Raw orders: 99441 | Analytical table: 96470


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_days,is_late,purchase_year,purchase_month,purchase_dayofweek,purchase_hour,n_items,n_products,n_sellers,total_price,total_freight,total_weight_g,total_volume_cm3,main_category,seller_state,total_payment,max_installments,n_payment_methods,main_payment_type,review_score,customer_unique_id,customer_state,customer_city,customer_zip_code_prefix
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,0,2017,10,0,10,1,1,1,29.99,8.72,500.0,1976.0,housewares,SP,38.71,1.0,2.0,voucher,4.0,7c396fd4830fd04220f754e42b4e5bff,SP,sao paulo,3149
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,0,2018,7,1,20,1,1,1,118.70,22.76,400.0,4693.0,perfumery,SP,141.46,1.0,1.0,boleto,4.0,af07308b275d755c9edb36a90c618231,BA,barreiras,47813
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,0,2018,8,2,8,1,1,1,159.90,19.22,420.0,9576.0,auto,SP,179.12,3.0,1.0,credit_card,5.0,3a653a41f6f9fc3d2a113cf8398680e8,GO,vianopolis,75265
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,26.188819,0,2017,11,5,19,1,1,1,45.00,27.20,450.0,6000.0,pet_shop,MG,72.20,1.0,1.0,credit_card,5.0,7c142cf63193a1473d2e66489a9ae977,RN,sao goncalo do amarante,59296
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,12.112049,0,2018,2,1,21,1,1,1,19.90,8.72,250.0,11475.0,stationery,SP,28.62,1.0,1.0,credit_card,5.0,72632f0f9dd73dfee390c9b22eb56dd6,SP,santo andre,9195


2. Target balance

In [2]:
print(master["is_late"].value_counts())
print(master["is_late"].value_counts(normalize=True).round(3))

is_late
0    88644
1     7826
Name: count, dtype: int64
is_late
0    0.919
1    0.081
Name: proportion, dtype: float64


3. Sanity checks

In [3]:
print("Duplicate order_ids:", master["order_id"].duplicated().sum())
print("Negative delivery_days:", (master["delivery_days"] < 0).sum())
master[["delivery_days", "estimated_days", "total_price", "total_freight",
        "total_weight_g", "total_volume_cm3"]].describe().round(2)

Duplicate order_ids: 0
Negative delivery_days: 0


,delivery_days,estimated_days,total_price,total_freight,total_weight_g,total_volume_cm3
count,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00
mean,12.56,23.74,137.04,22.79,2386.47,17329.06
std,9.55,8.76,209.05,21.56,4769.81,30142.20
min,0.53,2.01,0.85,0.00,0.00,0.00
25%,6.77,18.33,45.90,13.85,300.00,2964.00
50%,10.22,23.23,86.50,17.17,750.00,7250.00
75%,15.72,28.41,149.90,24.02,2050.00,19866.00
max,209.63,155.14,13440.00,1794.96,184400.00,1476000.00


4. Missing values after the join

In [4]:
missing = master.isna().sum()
missing = missing[missing > 0]
pd.DataFrame({"missing": missing, "pct": (missing / len(master) * 100).round(2)})

,missing,pct
order_approved_at,14,0.01
order_delivered_carrier_date,1,0.00
total_payment,1,0.00
max_installments,1,0.00
n_payment_methods,1,0.00
main_payment_type,1,0.00
review_score,646,0.67


In [ ]:
(master.groupby("customer_state")["is_late"]
       .agg(orders="count", late_rate="mean")
       .query("orders >= 500")
       .sort_values("late_rate", ascending=False)
       .round(3))

5. Save

In [5]:
out_path = PROJECT_ROOT / "data" / "processed" / "orders_master.parquet"
master.to_parquet(out_path, index=False)
print("Saved", out_path, master.shape)

Saved c:\Users\zikil\Documents\Fatima\projects\olist-ecommerce-analytics\data\processed\orders_master.parquet (96470, 33)


6. Splitting training and test data

In [6]:
from src.data_prep import time_split

train, test = time_split(master, test_size=0.2)
for name, part in [("Train", train), ("Test", test)]:
    print(f"{name}: {part.shape} | {part['order_purchase_timestamp'].min().date()} "
          f"to {part['order_purchase_timestamp'].max().date()} "
          f"| late rate {part['is_late'].mean():.3f}")

Train: (77176, 33) | 2016-09-15 to 2018-05-26 | late rate 0.088
Test: (19294, 33) | 2018-05-26 to 2018-08-29 | late rate 0.053


In [7]:
processed = PROJECT_ROOT / "data" / "processed"
train.to_parquet(processed / "train.parquet", index=False)
test.to_parquet(processed / "test.parquet", index=False)